In [1]:
import torch
import random

In [2]:
words = open("names.txt", "r").read().splitlines()

# create vocabulary
chars = sorted(list(set("".join(words))))

stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0

itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(stoi)

print("total words:", len(words))
print("vocab size :", VOCAB_SIZE)

total words: 32033
vocab size : 27


In [3]:
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_words = words[:n1]
dev_words   = words[n1:n2]
test_words  = words[n2:]

print("\nDataset Split")
print("train:", len(train_words))
print("dev  :", len(dev_words))
print("test :", len(test_words))


Dataset Split
train: 25626
dev  : 3203
test : 3204


In [4]:
counts = torch.zeros((VOCAB_SIZE, VOCAB_SIZE, VOCAB_SIZE), dtype=torch.int32)

for word in train_words:
    
    chars_in_word = [".", "."] + list(word) + ["."]
    
    for ch1, ch2, ch3 in zip(chars_in_word, chars_in_word[1:], chars_in_word[2:]):
        
        i = stoi[ch1]
        j = stoi[ch2]
        k = stoi[ch3]
        
        counts[i, j, k] += 1

In [5]:
def build_probability_table(alpha):
    
    probs = (counts + alpha).float()
    probs /= probs.sum(dim=2, keepdim=True)
    
    return probs

In [6]:
def evaluate_loss(split_words, probs):
    
    log_likelihood = 0.0
    total_predictions = 0
    
    for word in split_words:
        
        chars_in_word = [".", "."] + list(word) + ["."]
        
        for ch1, ch2, ch3 in zip(chars_in_word, chars_in_word[1:], chars_in_word[2:]):
            
            i = stoi[ch1]
            j = stoi[ch2]
            k = stoi[ch3]
            
            p = probs[i, j, k]
            
            log_likelihood += torch.log(p)
            total_predictions += 1
    
    loss = -log_likelihood / total_predictions
    return loss.item()


In [7]:
alphas = [
    0.0001,
    0.001,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]

best_alpha = None
best_dev_loss = float("inf")

results = []

print("\nalpha\t\ttrain_loss\tdev_loss")
print("-" * 45)

for alpha in alphas:
    
    probs = build_probability_table(alpha)
    
    train_loss = evaluate_loss(train_words, probs)
    dev_loss   = evaluate_loss(dev_words, probs)
    
    results.append((alpha, train_loss, dev_loss))
    
    print(f"{alpha:<8}\t{train_loss:.4f}\t\t{dev_loss:.4f}")
    
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_alpha = alpha




alpha		train_loss	dev_loss
---------------------------------------------
0.0001  	2.1830		2.2580
0.001   	2.1830		2.2447
0.01    	2.1835		2.2318
0.03    	2.1845		2.2264
0.1     	2.1877		2.2225
0.3     	2.1950		2.2234
1.0     	2.2157		2.2365
3.0     	2.2586		2.2727
10.0    	2.3563		2.3637


In [8]:
best_probs = build_probability_table(best_alpha)
test_loss = evaluate_loss(test_words, best_probs)

print("\n" + "=" * 50)
print("BEST SMOOTHING FOUND")
print("=" * 50)
print("best alpha :", best_alpha)
print("best dev   :", round(best_dev_loss, 4))
print("test loss  :", round(test_loss, 4))

# If alpha is very small train loss becomes low, dev loss may worse -> overfitting
# If alpha is very large, train and dev loss rises -> too much smoothing
# Middle alpha: best generalization


BEST SMOOTHING FOUND
best alpha : 0.1
best dev   : 2.2225
test loss  : 2.224


In [ ]:
#SAMPLE NAMES USING BEST ALPHA

g = torch.Generator().manual_seed(2147483647)

print("\nGenerated names:\n")

for _ in range(10):
    
    out = []
    i, j = 0, 0
    
    while True:
        
        p = best_probs[i, j]
        k = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        
        if k == 0:
            break
        
        out.append(itos[k])
        
        i, j = j, k
    
    print("".join(out))